# 04 — Real Data Ingestion + Hybrid Pipeline

End-to-end pipeline: real + Kenya-context synthetic data → GNN training (both domains).

## What runs here

| Step | Data source | Pipeline output |
|------|-------------|-----------------|
| 1 | CISA KEV + FIRST EPSS (live API) | `VULNERABILITY_EVENT` rows |
| 2 | CIC-IDS2018 traffic (real CSV or auto-generated Kenya synthetic) | `DDOS_SIGNAL_EVENT` + `WEB_ATTACK_EVENT` rows |
| 3 | CAIDA DDoS traces (real CSV or auto-generated Kenya synthetic) | `DDOS_SIGNAL_EVENT` rows |
| 4 | Kenya procurement fraud synthetic seed | `graph_feature_snapshot` rows (`Wcorruption`) |
| 5 | Feature snapshot worker | updates `Wmid` and `Wcorruption` windows |
| 6 | Cyber GNN train | artifact saved to `/app/artifacts/gnn/` |
| 7 | Corruption GNN train | artifact saved to `/app/artifacts/gnn/` |

## Data flow
```
raw rows
  → app.integrations.real_data_pipeline normalizers
  → connector events
  → event_log + event_entity_index
  → graph_feature_snapshot (Wmid / Wcorruption)
  → GNN train/eval
```

## Real datasets (optional — pipeline uses Kenya synthetic if not present)

| Dataset | URL | Use |
|---------|-----|-----|
| CIC-IDS2018 | https://www.kaggle.com/datasets/solarmainframe/ids-intrusion-csv | DDoS + Web attacks |
| CAIDA DDoS 2007 | https://www.caida.org/catalog/datasets/ddos-20070804_dataset/ | Volumetric DDoS (registration required) |
| PaySim mobile money | https://www.kaggle.com/datasets/ntnu-testimon/paysim1 | M-Pesa fraud EDA |
| CISA KEV + EPSS | Live APIs (no download) | Vulnerability events ✅ already wired |

## Inputs you control

- `SOURCE_API_KEY` — use a seeded source key (default `safaricom-secret-key` works with `seed_default_sources()`)
- `ASSET_ID` — target asset ID for KEV vulnerability events
- `CIC_INPUT_FILE` — path to real CIC-IDS2018 CSV **or** `None` to auto-generate Kenya synthetic data
- `CAIDA_INPUT_FILE` — path to real CAIDA traces CSV **or** `None` to auto-generate Kenya synthetic data

**To use real CIC data:** download from Kaggle, unzip, set `CIC_INPUT_FILE = '/path/to/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv'`

**To use real CAIDA data:** register at caida.org, download `ddos-20070804`, convert to CSV, set `CAIDA_INPUT_FILE = '/path/to/caida_rows.csv'`

All generated synthetic files go into `notebooks/data/` which is git-ignored.

In [ ]:
import sys
from pathlib import Path

HERE = Path.cwd()
NOTEBOOKS_DIR = HERE if HERE.name == 'notebooks' else (HERE / 'notebooks')
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

from pipeline_bootstrap import (
    bootstrap_environment,
    check_notebook_prerequisites,
    repair_notebook_schema,
    event_type_counts_last_24h,
    ingest_kev_epss,
    ingest_traffic_file,
    ingest_paysim_file,
    ingest_ocds_file,
    apply_feedback_to_snapshots,
    retrain_with_feedback,
    query_active_learning,
    run_feature_snapshots,
    seed_default_sources,
    train_gnn,
)
from seed_realistic_data import (
    generate_cic_csv,
    generate_caida_csv,
    generate_paysim_csv,
    generate_ocds_json,
)

env = bootstrap_environment()
print("Environment:", env)

pre = check_notebook_prerequisites()
if not pre.get('ok'):
    print("Schema issues detected — auto-repairing...")
    print(repair_notebook_schema())

seed_default_sources()
print("Sources seeded.")

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
SOURCE_API_KEY = 'safaricom-secret-key'
ASSET_ID       = 'county-finance-db-01'

# CISA KEV + EPSS: always fetched live — no file needed
RUN_KEV = True

# CIC traffic: set to a real CIC-IDS2018 CSV path, or None for Kenya synthetic
CIC_INPUT_FILE = None    # e.g. '/data/cicids2018/Friday-DDos.csv'

# CAIDA traffic: set to a real CAIDA traces CSV path, or None for Kenya synthetic
CAIDA_INPUT_FILE = None  # e.g. '/data/caida/ddos-20070804-rows.csv'

# PaySim M-Pesa mobile money fraud: None → auto-generate Kenya synthetic
# Real data: https://www.kaggle.com/datasets/ntnu-testimon/paysim1
PAYSIM_INPUT_FILE = None  # e.g. '/data/PS_20174392719_1491204439457_log.csv'
RUN_PAYSIM = True

# OCDS Kenya procurement: None → auto-generate Kenya synthetic
# Real data: https://ocds.open-contracting.org/
OCDS_INPUT_FILE = None    # e.g. '/data/kenya_ocds_releases.json'
RUN_OCDS = True

# Corruption pipeline
RUN_CORRUPTION_SEED  = True   # seed Kenya procurement fraud patterns into DB
RUN_CORRUPTION_TRAIN = True   # train corruption GNN after seeding

# Feedback loop: apply analyst-accepted AIFeedbackLabel rows before retraining
RUN_FEEDBACK = True  # set False to skip feedback application

In [ ]:
# ── Auto-generate synthetic files when real data not provided ─────────────────
# generate_cic_csv / generate_caida_csv produce Kenya-context CSVs whose column
# schemas exactly match the real CIC-IDS2018 / CAIDA normalizers, so the ingest
# step works identically whether you supply real data or synthetic data.
#
# To use real data instead, set the paths in the config cell above before running.

import os

DATA_DIR = os.path.join(os.path.dirname(os.path.abspath('.')), 'notebooks', 'data') \
    if os.path.basename(os.getcwd()) != 'notebooks' else 'data'

if CIC_INPUT_FILE is None:
    print("CIC_INPUT_FILE not set — generating Kenya synthetic CIC data...")
    CIC_INPUT_FILE = generate_cic_csv(
        os.path.join(DATA_DIR, 'kenya_cic_synthetic.csv'),
        n_rows=3_000,
    )
    print(f"  Generated: {CIC_INPUT_FILE}")
else:
    print(f"CIC_INPUT_FILE: {CIC_INPUT_FILE}")

if CAIDA_INPUT_FILE is None:
    print("CAIDA_INPUT_FILE not set — generating Kenya synthetic CAIDA data...")
    CAIDA_INPUT_FILE = generate_caida_csv(
        os.path.join(DATA_DIR, 'kenya_caida_synthetic.csv'),
        n_rows=800,
    )
    print(f"  Generated: {CAIDA_INPUT_FILE}")
else:
    print(f"CAIDA_INPUT_FILE: {CAIDA_INPUT_FILE}")

In [ ]:
# ── Auto-generate PaySim + OCDS synthetic files when real data not provided ───
# PaySim produces M-Pesa-style fraud chains (TRANSFER→TRANSFER→CASH_OUT).
# OCDS produces Open Contracting Data Standard releases with Kenya procurement
# corruption patterns (single-source, director conflict, price inflation, etc.).

if RUN_PAYSIM and PAYSIM_INPUT_FILE is None:
    print("PAYSIM_INPUT_FILE not set — generating Kenya synthetic PaySim data...")
    PAYSIM_INPUT_FILE = generate_paysim_csv(
        os.path.join(DATA_DIR, 'kenya_paysim_synthetic.csv'),
        n_steps=150,
    )
    print(f"  Generated: {PAYSIM_INPUT_FILE}")
elif RUN_PAYSIM:
    print(f"PAYSIM_INPUT_FILE: {PAYSIM_INPUT_FILE}")

if RUN_OCDS and OCDS_INPUT_FILE is None:
    print("OCDS_INPUT_FILE not set — generating Kenya synthetic OCDS releases...")
    OCDS_INPUT_FILE = generate_ocds_json(
        os.path.join(DATA_DIR, 'kenya_ocds_synthetic.json'),
        n_releases=500,
    )
    print(f"  Generated: {OCDS_INPUT_FILE}")
elif RUN_OCDS:
    print(f"OCDS_INPUT_FILE: {OCDS_INPUT_FILE}")

In [ ]:
# ── Ingest cyber events (KEV + CIC + CAIDA) ───────────────────────────────────
results = []

if RUN_KEV:
    print("Ingesting CISA KEV + FIRST EPSS (live API)...")
    results.append(ingest_kev_epss(source_api_key=SOURCE_API_KEY, asset_id=ASSET_ID))
    print("  →", results[-1])

# CIC file is always available at this point (real or synthetic)
print("Ingesting CIC traffic data...")
results.append(
    ingest_traffic_file(
        dataset='cic',
        input_file=CIC_INPUT_FILE,
        source_api_key=SOURCE_API_KEY,
        service_id_prefix='kenya-infra',
        dataset_name='cic_ids2018_kaggle',
    )
)
print("  →", results[-1])

# CAIDA file is always available at this point (real or synthetic)
print("Ingesting CAIDA DDoS data...")
results.append(
    ingest_traffic_file(
        dataset='caida',
        input_file=CAIDA_INPUT_FILE,
        source_api_key=SOURCE_API_KEY,
        service_id_prefix='kenya-infra',
        dataset_name='caida_ddos',
    )
)
print("  →", results[-1])

results

In [ ]:
# ── Ingest PaySim M-Pesa fraud chains (cyber domain) ─────────────────────────
# Only isFraud=1 rows with CASH_OUT/TRANSFER/DEBIT are ingested.
# They enter the graph as TRANSACTION_EVENTs via core_banking_tx_v1 connector,
# building mule-chain graph structure the Cyber GNN learns from.

if RUN_PAYSIM:
    print("Ingesting PaySim M-Pesa fraud chains...")
    paysim_result = ingest_paysim_file(
        input_file=PAYSIM_INPUT_FILE,
        source_api_key=SOURCE_API_KEY,
        dataset_name='paysim_ke',
    )
    print("  →", paysim_result)
else:
    paysim_result = {}
    print("PaySim ingest skipped (RUN_PAYSIM=False)")

# ── Ingest OCDS Kenya procurement releases (corruption domain) ────────────────
# OCDS releases are aggregated per-party and written directly to
# graph_feature_snapshot (window_key=Wcorruption) — bypassing event_log
# because the Corruption GNN reads directly from graph_feature_snapshot.

if RUN_OCDS:
    print("Ingesting OCDS Kenya procurement releases...")
    ocds_result = ingest_ocds_file(
        input_file=OCDS_INPUT_FILE,
        source_api_key=SOURCE_API_KEY,
    )
    print("  →", ocds_result)
else:
    ocds_result = {}
    print("OCDS ingest skipped (RUN_OCDS=False)")

{'paysim': paysim_result, 'ocds': ocds_result}

In [ ]:
# ── Seed corruption domain data (large-scale) ──────────────────────────────────
# Inserts ~3,500 Kenya procurement fraud nodes into graph_feature_snapshot
# (window_key="Wcorruption") via 10 community runs × 3 families:
#   Tender Cartel · Ghost Workers · Shell Company Network + 100 benign/run
#
# Uses seed_large_scale for consistency with the dashboard "Seed Corruption Data"
# button — both routes now produce the same scale of training data.

if RUN_CORRUPTION_SEED:
    print("Seeding large-scale Kenya corruption patterns (~3,500 nodes)...")
    from app.demo.seed_large_scale import seed_corruption as _seed_corruption_large
    corruption_seed_result = _seed_corruption_large(n_runs=10, benign_per_run=100)
    print("Corruption seed complete:", corruption_seed_result)
else:
    corruption_seed_result = {}
    print("Corruption seed skipped (RUN_CORRUPTION_SEED=False)")

In [ ]:
# ── Build feature snapshots for both windows ──────────────────────────────────
# Wmid       → cyber GNN input (vulnerability + DDoS + web attack events)
# Wcorruption → corruption GNN input (already built by seed_corruption above,
#               re-run here to pick up any new events from the cyber ingest)

print("Building Wmid feature snapshots (cyber)...")
cyber_stats = run_feature_snapshots(window_keys=('Wmid',), max_entities=6_000)
print("  Wmid:", cyber_stats)

print("Building Wcorruption feature snapshots (corruption)...")
corruption_stats = run_feature_snapshots(window_keys=('Wcorruption',), max_entities=6_000)
print("  Wcorruption:", corruption_stats)

{'cyber': cyber_stats, 'corruption': corruption_stats}

In [ ]:
# ── Train Cyber GNN ───────────────────────────────────────────────────────────
print("Training Cyber GNN (window=Wmid, epochs=60)...")
cyber_train = train_gnn(domain='cyber', epochs=60)
print("Cyber GNN result:", cyber_train)

In [ ]:
# ── Train Corruption GNN ──────────────────────────────────────────────────────
if RUN_CORRUPTION_TRAIN:
    print("Training Corruption GNN (window=Wcorruption, epochs=60)...")
    corruption_train = train_gnn(domain='corruption', epochs=60)
    print("Corruption GNN result:", corruption_train)
else:
    corruption_train = {}
    print("Corruption GNN training skipped (RUN_CORRUPTION_TRAIN=False)")

In [ ]:
# ── Analyst Feedback Loop ─────────────────────────────────────────────────────
# If analysts have accepted AIFeedbackLabel corrections (via the dashboard UI),
# this cell stamps those labels into graph_feature_snapshot as risk_flags, then
# retrains both GNNs on the corrected snapshots.
#
# When no feedback labels exist yet this is a fast no-op (0 rows updated).
# Run this cell after any analyst review session to close the correction loop.

if RUN_FEEDBACK:
    print("Applying analyst feedback labels to cyber snapshots...")
    feedback_cyber = apply_feedback_to_snapshots(domain='cyber', window_key='Wmid')
    print("  Cyber feedback:", feedback_cyber)

    print("Applying analyst feedback labels to corruption snapshots...")
    feedback_corruption = apply_feedback_to_snapshots(domain='corruption', window_key='Wcorruption')
    print("  Corruption feedback:", feedback_corruption)

    if feedback_cyber['updated'] > 0 or feedback_corruption['updated'] > 0:
        print(f"  {feedback_cyber['updated'] + feedback_corruption['updated']} snapshots updated — retraining GNNs...")
        retrain_cyber = train_gnn(domain='cyber', epochs=60)
        retrain_corruption = train_gnn(domain='corruption', epochs=60)
        print("  Cyber retrain:", retrain_cyber)
        print("  Corruption retrain:", retrain_corruption)
    else:
        print("  No new feedback labels — skipping retrain.")
        retrain_cyber = retrain_corruption = {}

    feedback_summary = {
        'cyber_feedback': feedback_cyber,
        'corruption_feedback': feedback_corruption,
        'cyber_retrain': retrain_cyber,
        'corruption_retrain': retrain_corruption,
    }
else:
    feedback_summary = {}
    print("Feedback loop skipped (RUN_FEEDBACK=False)")

feedback_summary

## Active Learning — Priority Queue for Analyst Review

The GNN uses **MC Dropout** to estimate prediction uncertainty for every entity. `query_active_learning()` surfaces the nodes where the model is *least confident* — these are the highest-value annotations an analyst can provide.

**Workflow:**
1. Run the cell below after training to get the top-50 uncertain entities
2. Review them in the Sentinel-KE dashboard (flag / clear)
3. Run the Feedback Loop cell to stamp accepted labels → retrain

This mirrors the RLHF pipeline at Anthropic/OpenAI: human feedback is concentrated on the model's *uncertainty frontier* rather than random sampling.

In [ ]:
# ── Active Learning: surface highest-uncertainty entities for analyst review ───
# Run this after training. Results are sorted by uncertainty (MC Dropout std).
# uncertainty=1.0 → model has no idea; uncertainty=0.0 → model is certain.
# Review the top entries in the dashboard and accept/reject the predictions.

print("=== Top uncertain entities — CYBER domain (Wmid) ===")
cyber_uncertain = query_active_learning(domain='cyber', window_key='Wmid', top_k=20)
for i, row in enumerate(cyber_uncertain[:10], 1):
    u = row['uncertainty']
    s = row['score']
    bar = '█' * int(u * 20) + '░' * (20 - int(u * 20))
    print(f"  {i:2d}. [{bar}] u={u:.3f} score={s:.3f}  {row['entity_key'][:48]}")

print(f"\n=== Top uncertain entities — CORRUPTION domain (Wcorruption) ===")
corruption_uncertain = query_active_learning(domain='corruption', window_key='Wcorruption', top_k=20)
for i, row in enumerate(corruption_uncertain[:10], 1):
    u = row['uncertainty']
    s = row['score']
    bar = '█' * int(u * 20) + '░' * (20 - int(u * 20))
    print(f"  {i:2d}. [{bar}] u={u:.3f} score={s:.3f}  {row['entity_key'][:48]}")

print(f"\nTotal queued for review: cyber={len(cyber_uncertain)}, corruption={len(corruption_uncertain)}")
print("Review these in the dashboard, then re-run the Feedback Loop cell above.")

{'cyber_top_uncertain': cyber_uncertain, 'corruption_top_uncertain': corruption_uncertain}

In [ ]:
# ── Pipeline summary ──────────────────────────────────────────────────────────
summary = {
    'ingest_cyber':       results,
    'ingest_paysim':      paysim_result,
    'ingest_ocds':        ocds_result,
    'cyber_features':     cyber_stats,
    'corruption_features': corruption_stats,
    'cyber_gnn':          cyber_train,
    'corruption_gnn':     corruption_train,
    'feedback':           feedback_summary,
}

print("\n=== Pipeline complete ===")
for k, v in summary.items():
    print(f"  {k}: {v}")

summary

In [ ]:
feature_stats = run_feature_snapshots(window_keys=('Wmid',), max_entities=6000)
feature_stats


In [ ]:
train_result = train_gnn(domain='cyber', epochs=60)
train_result
